# 06｜跳跃：参考动作、强化学习与评估

用上游已公开的跳跃参考数据与 PPO 任务，在仿真中训练和回放跳跃。结果及视频读取本专题保存的实测文件；没有运行的数据不会被写成成果。

## 运行前提

本文记录的上游代码来自 [KingKongRobotics/jumper](https://github.com/KingKongRobotics/jumper)，提交 `b2e58669848a3ba4f209597cd6bebdfe9d0533a4`。完整环境检查和旧实验见 [Ubuntu 工作站复现记录](../01-学习资料/Ubuntu工作站复现记录.md)。下方代码单元读取已经保存的结果并显示视频，不会自动安装依赖或启动训练。

工作站代码在 `/home/ubuntu/projects/robotics_shared/repos/jumper-everyembodied`，Python 为 `/data/Data14TB/envs/jumper-everyembodied/bin/python`。下方命令在仓库根目录执行；`warp:cuda` 用于 GPU 训练，`native:cpu` 和 OSMesa 用于无窗口录像。训练日志、checkpoint、视频和评估数据来自不同步骤，应在报告中分别记录。具体版本和结果以本专题保存的证据为准。


## 1. 开源内容和动作机制

[训练任务与奖励](https://github.com/KingKongRobotics/jumper/tree/b2e58669848a3ba4f209597cd6bebdfe9d0533a4/tasks/jumper/jump) 使用仓库内 `tasks/jumper/jump/ref/high_jump_flat.npz`。策略输出是在参考关节动作上的残差；学习目标包含参考关节、机身高度、速度及接触跟踪。上游 PPO 网络与超参数见该目录的 `rl_cfg.py`。参考轨迹不等于当前训练出的策略；不要把参考动画或官方动作包当作这次 checkpoint 的结果。

## 2. 列出任务并训练

```bash
cd /home/ubuntu/projects/robotics_shared/repos/jumper-everyembodied
PY=/data/Data14TB/envs/jumper-everyembodied/bin/python
"$PY" scripts/train.py --task jumper.jump --dry-run
OMP_NUM_THREADS=4 "$PY" scripts/train.py --task jumper.jump --backend warp --device cuda:0 --num_envs 4096 --max-iterations 1000 --headless --no-tensorboard
```

以训练实际日志为准记录源码 commit、环境、任务、后端、环境数、迭代、随机种子和 checkpoint。上游默认训练可能超过此处示例的 1000 次；短训用于验证学习与运行链路，不自动表示收敛。

本次计划 1000 次，但工作站与其他任务共享 GPU，在保存 model_300 后主动中止；评估使用 model_300，上游默认 10000 次。

## 3. 阅读训练过程

除了总奖励，关注 `Episode_Reward/ref_base_height`、`ref_joint_pos`、`ref_contact` 以及 `Metrics/jump/apex_rise`、`flight_s`、`takeoff_vz`。训练日志中的批量统计与单次视频指标是不同测量。需要检查机器人实际离地、落地及失败终止，才能描述回放效果。

## 4. 回放、录像与上游评估

跳跃训练物理频率 400 Hz、控制频率 200 Hz；回放使用相同物理频率。观测噪声默认开启，`--stop-on-done` 避免将重置后的画面误认为连续动作。

```bash
CKPT=logs/jumper/jumper.jump/2026-10-29_14-48-03/model_300.pt
MUJOCO_GL=osmesa OMP_NUM_THREADS=1 "$PY" scripts/play.py --task jumper.jump --backend native --device cpu --checkpoint "$CKPT" --headless --steps 700 --physics-hz 400 --stop-on-done --measure --video jump.mp4 --video-width 640 --video-height 480 --speed 0
"$PY" tools/checks/jump_eval.py --ckpt "$CKPT" --num-envs 1 --episodes 3
```

上游评估脚本关闭观测噪声，以 1 N 接触力阈值判断全足离地，报告高度与累计腾空时间；它在自动重置后采样，且把一次 done 波计为一个 episode，本例用单环境解释其输出。训练批量指标与单次回放指标不同。


## 5. 本专题实测数据

In [1]:
import json
from pathlib import Path
from IPython.display import Video, display

evidence_dir = Path('../01-学习资料')
media_dir = Path('../assets')


In [2]:
result_path = evidence_dir / 'jump-result.json'
if result_path.is_file():
    result = json.loads(result_path.read_text(encoding="utf-8"))
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("尚未找到实际评估数据:", result_path)


{
  "task": "jumper.jump",
  "upstream_commit": "b2e58669848a3ba4f209597cd6bebdfe9d0533a4",
  "training": {
    "backend": "warp:cuda",
    "num_envs": 4096,
    "planned_iterations": 1000,
    "checkpoint_iteration": 300,
    "stopped_early": true,
    "seed": null
  },
  "checkpoint": "../02-可运行代码/checkpoints/jump-model-300.pt",
  "checkpoint_sha256": "03a568f4e0201bd5b1be977fbd8c94b5d9d9c88f9411cfa4e0ac9c4caeaec9bc",
  "evaluation": {
    "tool": "tools/checks/jump_eval.py",
    "observation_noise": false,
    "num_envs": 1,
    "episodes_reported": 3,
    "threshold": "All foot contact force norms <= 1 N",
    "reference_apex_z_m": 0.235,
    "reference_rise_m": 0.141,
    "reference_airborne_s": 0.319,
    "episodes": [
      {
        "apex_z_m": 0.254,
        "rise_m": 0.159,
        "airborne_s": 0.325,
        "push_off_vz_m_s": 1.6
      },
      {
        "apex_z_m": 0.252,
        "rise_m": 0.158,
        "airborne_s": 0.32,
        "push_off_vz_m_s": 1.6
      },
      {


## 6. 自训策略视频

下方视频和评估文件应对应同一 checkpoint 与明确的回放条件。

In [3]:
video_path = media_dir / 'jump-trained.mp4'
if video_path.is_file():
    display(Video(str(video_path), embed=True, width=640))
else:
    print("尚未找到实际视频:", video_path)


## 延伸阅读

- [上游教程：三种技能训练与回放](https://github.com/KingKongRobotics/jumper/blob/b2e58669848a3ba4f209597cd6bebdfe9d0533a4/docs/TUTORIAL.md)
- [专题首页](../README.md) · [三足步态训练](03_动作训练与评估.ipynb) · [舞蹈任务](07_舞蹈_参考动作训练与评估.ipynb)

数据和源码的第三方许可见上游 `NOTICE`。仿真效果不直接代表真机效果。